In [ ]:
!pip install -q transformers==4.51.0 tokenizers==0.21.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
```python
# =========================================================
# VANILLA KNOWLEDGE DISTILLATION
# ViSoBERT Teacher -> Classic TextCNN Student
# Grid Search Temperature + Alpha
# =========================================================

import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from copy import deepcopy
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report


# =========================================================
# 1. CONFIG
# =========================================================

TRAIN_PATH = "TRAIN_PATH"
VAL_PATH   = "VAL_PATH"
TEST_PATH  = "TEST_PATH"

#Tải trực tiếp từ Hugging Face Hub 
TEACHER_PATH = "Hungdz06/vietnamese-toxic-speech-nphungzz"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LEN = 128
BATCH_SIZE = 64

EMBED_DIM = 300
FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 100
DROPOUT = 0.5

EPOCHS = 10
LR = 1e-3

TEMPERATURES = [1.5, 2.0, 2.5, 3.0, 4.0]
ALPHAS = [0.6, 0.7, 0.8, 0.9]

SEED = 42

BEST_MODEL_PATH = "/kaggle/working/textcnn_kd_best_grid.pt"
RESULTS_PATH = "/kaggle/working/kd_grid_results.csv"


# =========================================================
# 2. SEED
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print("=" * 70)
print("VANILLA KD GRID SEARCH")
print("=" * 70)
print("Device:", DEVICE)
print("Total experiments:", len(TEMPERATURES) * len(ALPHAS))


# =========================================================
# 3. LOAD DATA
# =========================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)


# =========================================================
# 4. BINARY LABEL
# =========================================================

def convert_binary(label):
    return 0 if label == 0 else 1


for df in [train_df, val_df, test_df]:
    df["binary_label"] = df["labels"].apply(convert_binary).astype(int)


print("\nTRAIN:")
print(train_df["binary_label"].value_counts())

print("\nVAL:")
print(val_df["binary_label"].value_counts())

print("\nTEST:")
print(test_df["binary_label"].value_counts())

print(f"\nTrain: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")


# =========================================================
# 5. TOKENIZER + VOCAB
# =========================================================

def tokenize(text):
    return re.findall(r"\w+", str(text).lower(), flags=re.UNICODE)


vocab = {"<PAD>": 0, "<UNK>": 1}

for text in train_df["texts"].fillna(""):
    for token in tokenize(text):
        if token not in vocab:
            vocab[token] = len(vocab)

print("\nVocab size:", len(vocab))


def encode_text(text):
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokenize(text)]
    ids = ids[:MAX_LEN]
    ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    return ids


# =========================================================
# 6. DATASET
# =========================================================

class TextDataset(Dataset):

    def __init__(self, df):
        self.texts = df["texts"].fillna("").astype(str).tolist()
        self.labels = df["binary_label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode_text(self.texts[idx]), dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.texts[idx]
        )


train_dataset = TextDataset(train_df)
val_dataset = TextDataset(val_df)
test_dataset = TextDataset(test_df)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)


# =========================================================
# 7. TEXTCNN
# =========================================================

class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=300,
        num_classes=2,
        filter_sizes=[2, 3, 4],
        num_filters=100,
        dropout=0.5
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k)
            for k in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(filter_sizes),
            num_classes
        )

    def forward(self, x):

        x = self.embedding(x)       # [B, L, E]
        x = x.permute(0, 2, 1)      # [B, E, L]

        features = []

        for conv in self.convs:
            h = F.relu(conv(x))
            h = F.max_pool1d(h, h.size(2)).squeeze(2)
            features.append(h)

        x = torch.cat(features, dim=1)
        x = self.dropout(x)

        return self.fc(x)


# =========================================================
# 8. LOAD TEACHER
# =========================================================

print("\nLoading ViSoBERT teacher...")

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_PATH)

teacher = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_PATH
).to(DEVICE)

teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False

print("Teacher loaded.")


# =========================================================
# 9. TEACHER LOGITS
# =========================================================

def get_teacher_logits(texts):

    inputs = teacher_tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        return teacher(**inputs).logits


# =========================================================
# 10. KD LOSS
# =========================================================

ce_loss = nn.CrossEntropyLoss()


def kd_loss(student_logits, teacher_logits, labels, temperature, alpha):

    loss_ce = ce_loss(student_logits, labels)

    student_log_probs = F.log_softmax(
        student_logits / temperature, dim=1
    )

    teacher_probs = F.softmax(
        teacher_logits / temperature, dim=1
    )

    loss_kd = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean"
    ) * temperature ** 2

    return alpha * loss_ce + (1 - alpha) * loss_kd


# =========================================================
# 11. EVALUATION
# =========================================================

def evaluate(model, loader):

    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():

        for input_ids, labels, texts in loader:

            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(input_ids)
            loss = ce_loss(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    toxic_f1 = f1_score(all_labels, all_preds, pos_label=1)
    avg_loss = total_loss / len(loader)

    return avg_loss, acc, macro_f1, toxic_f1


# =========================================================
# 12. TRAIN ONE CONFIG
# =========================================================

def train_one_config(temperature, alpha):

    print("\n" + "=" * 70)
    print(f"TEMPERATURE = {temperature}")
    print(f"ALPHA       = {alpha}")
    print("=" * 70)

    set_seed(SEED)

    student = TextCNN(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        num_classes=2,
        filter_sizes=FILTER_SIZES,
        num_filters=NUM_FILTERS,
        dropout=DROPOUT
    ).to(DEVICE)

    print(
        "Student parameters:",
        sum(p.numel() for p in student.parameters())
    )

    optimizer = torch.optim.Adam(student.parameters(), lr=LR)

    best_f1 = -1
    best_acc = 0
    best_toxic_f1 = 0
    best_epoch = 0
    best_state = None

    for epoch in range(EPOCHS):

        student.train()
        total_loss = 0

        for input_ids, labels, texts in train_loader:

            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            student_logits = student(input_ids)
            teacher_logits = get_teacher_logits(texts)

            loss = kd_loss(
                student_logits,
                teacher_logits,
                labels,
                temperature,
                alpha
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        val_loss, val_acc, val_f1, val_toxic_f1 = evaluate(
            student, val_loader
        )

        train_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Macro-F1: {val_f1:.4f} | "
            f"Val Toxic-F1: {val_toxic_f1:.4f}"
        )

        if val_f1 > best_f1:

            best_f1 = val_f1
            best_acc = val_acc
            best_toxic_f1 = val_toxic_f1
            best_epoch = epoch + 1
            best_state = deepcopy(student.state_dict())

    return {
        "Temperature": temperature,
        "Alpha": alpha,
        "Val Accuracy": best_acc,
        "Val Macro-F1": best_f1,
        "Val Toxic-F1": best_toxic_f1,
        "Best Epoch": best_epoch,
        "model_state": best_state
    }


# =========================================================
# 13. GRID SEARCH
# =========================================================

total_experiments = len(TEMPERATURES) * len(ALPHAS)

print("\n" + "=" * 70)
print("START GRID SEARCH")
print("=" * 70)
print("Temperatures:", TEMPERATURES)
print("Alphas:", ALPHAS)
print("Total experiments:", total_experiments)

results = []
best_result = None
best_model_state = None

experiment_number = 0


for temperature in TEMPERATURES:

    for alpha in ALPHAS:

        experiment_number += 1

        print("\n" + "#" * 70)
        print(f"EXPERIMENT {experiment_number}/{total_experiments}")
        print("#" * 70)

        result = train_one_config(temperature, alpha)

        results.append({
            "Temperature": result["Temperature"],
            "Alpha": result["Alpha"],
            "Val Accuracy": result["Val Accuracy"],
            "Val Macro-F1": result["Val Macro-F1"],
            "Val Toxic-F1": result["Val Toxic-F1"],
            "Best Epoch": result["Best Epoch"]
        })

        if (
            best_result is None
            or result["Val Macro-F1"] > best_result["Val Macro-F1"]
        ):

            best_result = result
            best_model_state = deepcopy(result["model_state"])

            torch.save({
                "model_state_dict": best_model_state,
                "temperature": result["Temperature"],
                "alpha": result["Alpha"],
                "val_accuracy": result["Val Accuracy"],
                "val_macro_f1": result["Val Macro-F1"],
                "val_toxic_f1": result["Val Toxic-F1"],
                "best_epoch": result["Best Epoch"],
                "vocab": vocab,
                "config": {
                    "MAX_LEN": MAX_LEN,
                    "EMBED_DIM": EMBED_DIM,
                    "FILTER_SIZES": FILTER_SIZES,
                    "NUM_FILTERS": NUM_FILTERS,
                    "DROPOUT": DROPOUT
                }
            }, BEST_MODEL_PATH)

            print("\n>>> NEW BEST CONFIG!")
            print(f">>> Temperature = {result['Temperature']}")
            print(f">>> Alpha = {result['Alpha']}")
            print(f">>> Val Macro-F1 = {result['Val Macro-F1']:.4f}")
            print(f">>> Val Toxic-F1 = {result['Val Toxic-F1']:.4f}")
            print(f">>> Best Epoch = {result['Best Epoch']}")
            print(f">>> Saved to: {BEST_MODEL_PATH}")


# =========================================================
# 14. RESULTS
# =========================================================

results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values("Val Macro-F1", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("GRID SEARCH RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

results_df.to_csv(RESULTS_PATH, index=False)

print("\nGrid results saved to:", RESULTS_PATH)


# =========================================================
# 15. BEST CONFIG
# =========================================================

best_row = results_df.iloc[0]

print("\n" + "=" * 70)
print("BEST CONFIGURATION")
print("=" * 70)

print(f"Temperature : {best_row['Temperature']}")
print(f"Alpha       : {best_row['Alpha']}")
print(f"Best Epoch  : {int(best_row['Best Epoch'])}")
print(f"Val Accuracy: {best_row['Val Accuracy']:.4f}")
print(f"Val Macro-F1: {best_row['Val Macro-F1']:.4f}")
print(f"Val Toxic-F1: {best_row['Val Toxic-F1']:.4f}")

print("\nBest model:", BEST_MODEL_PATH)


# =========================================================
# 16. LOAD BEST MODEL
# =========================================================

print("\n" + "=" * 70)
print("LOADING BEST MODEL FOR FINAL TEST")
print("=" * 70)

checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

student = TextCNN(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    num_classes=2,
    filter_sizes=FILTER_SIZES,
    num_filters=NUM_FILTERS,
    dropout=DROPOUT
).to(DEVICE)

student.load_state_dict(checkpoint["model_state_dict"])
student.eval()

print("Loaded Temperature:", checkpoint["temperature"])
print("Loaded Alpha:", checkpoint["alpha"])
print("Loaded Val Macro-F1:", checkpoint["val_macro_f1"])


# =========================================================
# 17. FINAL TEST
# =========================================================

all_preds = []
all_labels = []

with torch.no_grad():

    for input_ids, labels, texts in test_loader:

        input_ids = input_ids.to(DEVICE)

        logits = student(input_ids)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())


# =========================================================
# 18. TEST METRICS
# =========================================================

test_accuracy = accuracy_score(all_labels, all_preds)
test_macro_f1 = f1_score(all_labels, all_preds, average="macro")
test_toxic_f1 = f1_score(all_labels, all_preds, pos_label=1)

print("\n" + "=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)

print(f"Temperature : {checkpoint['temperature']}")
print(f"Alpha       : {checkpoint['alpha']}")
print(f"Accuracy    : {test_accuracy:.4f}")
print(f"Macro-F1    : {test_macro_f1:.4f}")
print(f"Toxic F1    : {test_toxic_f1:.4f}")


# =========================================================
# 19. CLASSIFICATION REPORT
# =========================================================

print("\n" + classification_report(
    all_labels,
    all_preds,
    target_names=["Clean", "Toxic"],
    digits=4
))


Device: cuda


FileNotFoundError: [Errno 2] No such file or directory: '/content/train_final.csv'